# Silver Layer

Objetivo:

Transformar los datos de Bronze para hacerlos consistentes, limpios y aptos para análisis. 
Las tareas específicas que se realizarán en esta capa serán las siguientes:

- Corregir tipos de datos
- Eliminar registros inválidos
- Desanidar estructuras JSON
- Estandarizar nombres
- Agregar metadata
- Generar Hash MD5
- Preparar el modelo analítico

## 1. Carguío de capa Bronze

In [0]:
# Leer capa Bronze

df_bronze = spark.table(
"workspace.bronze.world_bank_container_traffic"
)
display(df_bronze.limit(5))

indicator,country,countryiso3code,date,value,unit,obs_status,decimal
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2017,5.6577838324E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2016,5.2672486324E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2015,5.2329947324E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2014,5.0087043324E7,,,0
"List(IS.SHP.GOOD.TU, Container port traffic (TEU: 20 foot equivalent units))","List(XU, North America)",NAC,2013,4.8107682461E7,,,0


## 2. Creación de Silver Base

Para iniciar la creación de Silver se realizarán las siguientes tareas:

- Corregir tipos de datos
- Eliminar registros inválidos
- Desanidar JSON
- Estandarizar nombres

In [0]:
# Creamos un nuevo data frame corrigiendo los tipos de datos de los campos bronze "Value" y "Date" y desanidando los campos "country" e "indicator" 

from pyspark.sql.functions import col

df_silver = (

    df_bronze
    .filter(
        col("value").isNotNull()
    )
    .select(
        col("country.id")
            .alias("country_id"),
        col("country.value")
            .alias("country_name"),
        col("countryiso3code")
            .alias("country_iso3"),
        col("indicator.id")
            .alias("indicator_id"),
        col("indicator.value")
            .alias("indicator_name"),
        col("date")
            .cast("int")
            .alias("year"),
        col("value")
            .cast("double")
            .alias("port_traffic_teu")
    )
)

# 3. Validación de calidad del dataframe Silver

Para validar la calidad de los datos del dataframe generado, se realizarán las siguientes verificaciones a los campos correspondientes:

- Registros válidos
- Registros duplicados
- Registros nulos

In [0]:
#Contéo de Registros Válidos
df_silver.count()


2793

In [0]:
#Verificación de Nulos
from pyspark.sql.functions import *

df_silver.select(
    [
       count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in df_silver.columns
    ]
).show()

+----------+------------+------------+------------+--------------+----+----------------+
|country_id|country_name|country_iso3|indicator_id|indicator_name|year|port_traffic_teu|
+----------+------------+------------+------------+--------------+----+----------------+
|         0|           0|           0|           0|             0|   0|               0|
+----------+------------+------------+------------+--------------+----+----------------+



In [0]:
#Verificación de Duplicados

total_records = df_silver.count()
unique_records = df_silver.dropDuplicates().count()
duplicates = total_records - unique_records

print(f"Duplicados encontrados: {duplicates}")

Duplicados encontrados: 0


Los resultados obtenidos nos permiten asegurar que el dataframe generado:
- No presenta valores nulos
- NO Presenta valores duplicados

## 4. Creación de Hash MD5


In [0]:
from pyspark.sql.functions import (
    md5,
    concat_ws
)

df_silver = (
    df_silver
    .withColumn(
        "record_hash",
        md5(
            concat_ws(
                "|",    
            col("country_iso3"),
            col("year"),
            col("port_traffic_teu")
            )
        )
    )
)

## 5. Metadata de Carga

In [0]:
from pyspark.sql.functions import (
    current_timestamp
)

df_silver = (
    df_silver
    .withColumn(
        "silver_load_timestamp",
        current_timestamp()
    )
)

## 6. Almacenamiento  en Silver

In [0]:
#Creación del Esquema Silver
spark.sql(
"""
CREATE SCHEMA IF NOT EXISTS workspace.silver
"""
)

DataFrame[]

In [0]:
#Guardar en Silver
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.silver.port_traffic_clean"
    )
)

In [0]:
#Validamos que se hayan cargado todos los registros a la capa Silver
spark.table(
    "workspace.silver.port_traffic_clean"
).count()

2793

In [0]:
#Verificamos el Esquema de la capa
df_silver.printSchema()


root
 |-- country_id: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- country_iso3: string (nullable = true)
 |-- indicator_id: string (nullable = true)
 |-- indicator_name: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- port_traffic_teu: double (nullable = true)
 |-- record_hash: string (nullable = false)
 |-- silver_load_timestamp: timestamp (nullable = false)



El esquema de la capa nos muestra:

- Estructura JSON fué desanidada correctamente
- Se entandarizaron los nombres de las columnas
- Se corrigieron los tipos de datos
- Se resolivió el problema identificado durante el proceso de data profiling respecto al tipado del campo; convirtiendo el campo a "port_traffic_teu" de tipo decimal y eliminando los registros nulos de la métrica.
- Las fechas se convirtieron en el campo "year" tipo entero.
- Se generó el Hash MD5 
- Se generó el campo "timestamp" para verificar la trazabilidad de la carga 

In [0]:
# Visualizamos los 10 primeros registros del dataframe silver
display(df_silver.limit(10))

country_id,country_name,country_iso3,indicator_id,indicator_name,year,port_traffic_teu,record_hash,silver_load_timestamp
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2023,8917517.0,8bf63ccaca93829f850b9d6bdc9c5665,2026-09-22T17:18:54.191Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2022,8186999.0,d8733e84e9367373c421d3dd7cf7f652,2026-09-22T17:18:54.191Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2021,8567214.0,c6fd8daeaffd45e9fb5bc19dd9007f59,2026-09-22T17:18:54.191Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2020,8100808.0,e596a01c5258932dbf1cf87823e20b1d,2026-09-22T17:18:54.191Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2019,8562750.13,ac0b490f4c3e5c8ce2e5117060022e1e,2026-09-22T17:18:54.191Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2018,7919906.59,9da4c2d5d3e9994215c9622151ec6f96,2026-09-22T17:18:54.191Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2017,7328156.16,afa86e16d9ca7b99fd8815b333c7f904,2026-09-22T17:18:54.191Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2016,6464269.34,13992a9eea5a15fc9d5813bf5b200b48,2026-09-22T17:18:54.191Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2015,6722792.36,e3cd1079bc7da38b8847a705d2defa81,2026-09-22T17:18:54.191Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2014,6567432.71,57eeab2aeff37270c394534f784fb6d3,2026-09-22T17:18:54.191Z


Al visualizar los primeros registros de la tabla, se observa que el campo "country" aparentemente muestra la región y agrupación y no solo los países específicos. Para verificar esta suposición ejecutaremos la siguiente validación: 

In [0]:
# Validamos campo country

df_silver.select(
    "country_iso3",
    "country_name"
).distinct().show(
    50,
    truncate=False
)


+------------+-------------------------------------------------------------------------+
|country_iso3|country_name                                                             |
+------------+-------------------------------------------------------------------------+
|CSS         |Caribbean small states                                                   |
|LTE         |Late-demographic dividend                                                |
|LCN         |Latin America & Caribbean                                                |
|MNA         |Middle East, North Africa, Afghanistan & Pakistan (excluding high income)|
|TEA         |East Asia & Pacific (IDA & IBRD countries)                               |
|TEC         |Europe & Central Asia (IDA & IBRD countries)                             |
|EAP         |East Asia & Pacific (excluding high income)                              |
|IDA         |IDA total                                                                |
|CEB         |Central

El resultado obtenido de la última validación, muestra que el campo country muestra algunos registros con países "reales" y otros campos en los que se combinan Países, Regiones, Agrupaciones económicos: World, European Union, OECD. Por lo que, en la capa GOLD se definirá como procesar este campo. 